# Dictionnaires thématiques

On a cherché les thèmes qui reviennent dans la littérature. On crée des dictionnaires de mots à partir des termes mentionnés dans la littérature + complément avec LLM. 

Ensuite, à l'échelle d'une phrase, on crée un score pour chaque thème : nombre_mots_theme_n/nombre_mots_phrase.

Enfin, on regarde si : 
    - (1) les termes de la littérature sont bien présents ou non dans le corpus. Si cela varie selon les sports, le genre des athlètes ou des commentateur.ices.
    - (2) si on retrouve ou non la distribution genrée de ces thèmes (e)

Importer les données + pre-processing.

In [10]:
import pandas as pd
import os

data_paths = {
    "patin_art": "/home/onyxia/work/PESSD-GBOGC/PESSD/csv/patin_art/",
    "biathlon": "/home/onyxia/work/PESSD-GBOGC/PESSD/csv/biath/",
    "ski_alpin": "/home/onyxia/work/PESSD-GBOGC/PESSD/csv/ski_alp/",
    "ski_free": "/home/onyxia/work/PESSD-GBOGC/PESSD/csv/ski_free/",
    "curling": "/home/onyxia/work/PESSD-GBOGC/PESSD/csv/curl/",
    "patin_vit": "/home/onyxia/work/PESSD-GBOGC/PESSD/csv/patin_vit/"
}

metadata = pd.read_csv("/home/onyxia/work/PESSD-GBOGC/metadata.csv") 

df = pd.DataFrame()
for sport, data_path in data_paths.items():
    for file in os.listdir(data_path):
        if file.endswith(".csv"):
            temp = pd.read_csv(data_path + file)
            temp["sport"] = sport
            temp["ID"] = file.replace("_wav_transcription.csv", "").replace("_transcription.csv", "")  # ← nouveau
            df = pd.concat([df, temp])

df = df.reset_index(drop=True)
df = df.merge(metadata, on="ID", how="left")  # ← nouveau
print(f"Total : {len(df)} segments")
print(df["g_ath"].value_counts())

Total : 13286 segments
g_ath
H    6291
F    5060
M    1824
Name: count, dtype: int64


In [11]:
# Pre-processing
import re
import string

# Removing music and noise (only technical commentary)
df = df[(df["main_g"]=="female")|(df["main_g"]=="male")]

# text to sentences
df["text"] = df["text"].apply(lambda x : x.split(". "))
df = df.explode("text")

# Removes all words with capitals
df["text"] = df["text"].apply(lambda x : re.sub(r"\s*[A-Z]\w*\s*", " ", x).strip())

# Removes empty texts
df = df[df["text"].str.strip() != ""]

# Remove punctuation
df ["text"] = df["text"].apply(lambda x : x.translate(str.maketrans('', '', string.punctuation.replace("'", ""))))

In [12]:
print(df["sport"].unique())

<ArrowStringArray>
['patin_art', 'biathlon', 'ski_alpin', 'ski_free', 'curling', 'patin_vit']
Length: 6, dtype: str


Les thèmes sont : (M = si majoritairement masculin / F = si majoritairement féminin)

Issus de la littérature sur le genre des sportif.ves
- Forme physique (vitesse, puissance physique...) / H
- Force de caractère (résilience, combativité...) / H
- Mental + Compétitif/autoritaire/leadership / H
- Emotion / F
- Apparence physique / F
- Vie personnelle/passé / F
- Personnalité / F
- Chance / F
- Performance/Héroïsme (fantastique, légendaire...) / H
- Enjeux/Résultat (Qualification, victoire...) / H
- Age (Jeunesse...) / F
- Marqueur genré (femme, féminin...) / F
-Violence / M


 Issus de la littérature sur le genre des journalistes
- Coopération/équipe / F 
- Analytique (stats, etc) / M
- Encouragements / F
- Stratégie / M

In [13]:
# Dictionnaires

forme_physique = [
    "vitesse", "vite", "rapide","rapides", "puissant", "puissants","puissante", "puissantes","puissance", "puissamment",
    "fort", "forte", "fortes","force", "fortement","physique", "physiques","athlétique", "athlétiques","explosif",
    "explosive","explosifs",
    "explosives", "explosivité", "endurance" , "endurant", "endurants", "endurante", "endurantes", "résistance", 
    "résistant","résistants","résistante","résistantes","souffle", "essoufflé", "essoufflés", "essoufflée", "essoufflées",
    "condition", "forme", "musclé", "musclée", "musclés", "musclées","robuste", "robustes","robustesse",
    "vigoureux", "vigoureuse", "vigoureuses","énergie", "énergique", "énergiques","dynamique", "dynamiques", "dynamisme",
    "véloce", "véloces","vélocité", "détente", "agilité", "agile", "agiles","souplesse",
    "souple", "souples","léger", "légère", "légers", "légères","légèreté", "accélération", "sprint",
    "sprinter", "sprinté", "sprintée", "sprintés", "sprintées",
    "accélérer", "accélère", "accélères", "accélérons", "accélérez", "accélèrent",
    "accélérais", "accélérait", "accéléraient",
    "sprinte", "sprintes", "sprintons", "sprintez", "sprinent",
    "sprintais", "sprintait", "sprintaient",
    "bondir", "bondit", "bondis", "bondissons", "bondissez", "bondissent",
    "bondissais", "bondissait", "bondissaient",
    "foncer", "fonce", "fonces", "fonçons", "foncez", "foncent",
    "fonçais", "fonçait", "fonçaient",
    "courir", "court", "cours", "courons", "courez", "courent",
    "courais", "courait", "couraient",
]

force_caractere_effort = [
    # Résilience
    "résilience", "résilient", "résiliente", "résilients", "résilientes",
    "rebondir", "rebondit", "rebondis", "rebondissons", "rebondissez", "rebondissent",
    "rebondissais", "rebondissait", "rebondissaient",
    "surmonter", "surmonte", "surmontais", "surmontait", "surmontaient",
    "surmonté", "surmontée", "surmontés", "surmontées",
    "résister", "résiste", "résistes", "résistons", "résistez", "résistent",
    "résistais", "résistait", "résistaient",
    # Combativité
    "combatif", "combative", "combatifs", "combatives", "combativité",
    "batailleur", "batailleuse", "batailleurs", "batailleuses",
    "lutter", "lutte", "luttes", "luttons", "luttez", "luttent",
    "luttais", "luttait", "luttaient",
    "se battre", "se bat", "se battent", "se battait", "se battaient",
    "acharné", "acharnée", "acharnés", "acharnées", "acharnement",
    "tenace", "tenaces", "ténacité",
    "persévérant", "persévérante", "persévérants", "persévérantes", "persévérance",
    "persévérer", "persévère", "persévères", "persévèrent",
    "persévérais", "persévérait", "persévéraient",
    # Mental fort
    "mental", "mentale", "mentaux", "mentalement",
    "solide", "solides", "solidité",
    "déterminé", "déterminée", "déterminés", "déterminées", "détermination",
    "volonté", "volontaire", "volontaires",
    "courageux", "courageuse", "courageux", "courageuses", "courage",
    "cœur", "caractère",
    "inébranlable", "inébranlables",
    "imperturbable", "imperturbables",
    # Réaction après échec
    "réagir", "réagit", "réagis", "réagissons", "réagissez", "réagissent",
    "réagissais", "réagissait", "réagissaient",
    "relever", "relève", "relèves", "relevons", "relevez", "relèvent",
    "relevais", "relevait", "relevaient",
    "relevé", "relevée", "relevés", "relevées",
    # Effort
    "effort", "efforts",
    "travailler", "travaille", "travailles", "travaillent", "travaillait", "travaillaient",
    "travaillé", "travaillée", "travaillés", "travaillées", "travail",
    "s'entraîner", "s'entraîne", "s'entraînait", "s'entraînaient",
    "entraînement", "entraînements",
    "suer", "sue", "suait", "suaient", "sueur", "sueurs",
    "peiner", "peine", "peines", "peinait", "peinaient",
    "souffrir", "souffre", "souffrait", "souffraient", "souffrance", "souffrances",
    "sacrifier", "sacrifice", "sacrifices", "sacrifié", "sacrifiée", "sacrifiés", "sacrifiées",
    "mériter", "mérite", "mérites", "méritait", "méritaient",
    "mérité", "méritée", "mérités", "méritées",
    "donner", "donne", "donnait", "donnaient", "donné",
    "tout donner", "se donner",
    "impliquer", "impliqué", "impliquée", "impliqués", "impliquées", "implication",
    "investir", "investit", "investissait", "investissaient",
    "investi", "investie", "investis", "investies", "investissement",
    "appliquer", "applique", "appliquait", "appliquaient",
    "appliqué", "appliquée", "appliqués", "appliquées", "application",
]

mental_leadership = [
    # Mental / concentration
    "concentré", "concentrée", "concentrés", "concentrées", "concentration",
    "se concentrer", "se concentre", "se concentrait", "se concentraient",
    "focus", "focalisé", "focalisée", "focalisés", "focalisées",
    "lucide", "lucides", "lucidité",
    "calme", "calmes", "calmement",
    "sang-froid",
    "serein", "sereine", "sereins", "sereines", "sérénité",
    "maîtrise", "maîtriser", "maîtrise", "maîtrisait", "maîtrisaient",
    "maîtrisé", "maîtrisée", "maîtrisés", "maîtrisées",
    "intelligent", "intelligente", "intelligents", "intelligentes", "intelligemment",
    "tactique", "tactiques", "tactiquement",
    "réfléchi", "réfléchie", "réfléchis", "réfléchies",
    "réfléchir", "réfléchit", "réfléchissait", "réfléchissaient",
    # Leadership
    "leader", "leaders", "leadership",
    "capitaine", "capitaines",
    "mener", "mène", "mènes", "menons", "menez", "mènent",
    "menais", "menait", "menaient",
    "mené", "menée", "menés", "menées",
    "diriger", "dirige", "diriges", "dirigeons", "dirigez", "dirigent",
    "dirigeais", "dirigeait", "dirigeaient",
    "commandant", "commandante", "commandants", "commandantes",
    "patron", "patronne", "patrons", "patronnes",
    # Compétitif / autoritaire
    "compétitif", "compétitive", "compétitifs", "compétitives", "compétitivité",
    "ambitieux", "ambitieuse", "ambitieux", "ambitieuses", "ambition",
    "dominateur", "dominatrice", "dominateurs", "dominatrices",
    "dominer", "domine", "domines", "dominons", "dominez", "dominent",
    "dominais", "dominait", "dominaient",
    "dominé", "dominée", "dominés", "dominées",
    "imposer", "impose", "imposais", "imposait", "imposaient",
    "imposant", "imposante", "imposants", "imposantes",
    "autorité", "autoritaire", "autoritaires",
    "gagner", "gagne", "gagnes", "gagnons", "gagnez", "gagnent",
    "gagnais", "gagnait", "gagnaient",
    "gagneur", "gagneuse", "gagneurs", "gagneuses",
    "vainqueur", "vainqueurs",
]

emotion = [
    # Joie / bonheur
    "joie", "joyeux", "joyeuse", "joyeux", "joyeuses", "joyeusement",
    "heureux", "heureuse", "heureux", "heureuses", "heureusement", "bonheur",
    "content", "contente", "contents", "contentes", "contentement",
    "ravi", "ravie", "ravis", "ravies",
    "exulter", "exulte", "exultes", "exultent", "exultait", "exultaient",
    "exultant", "exultante", "exultants", "exultantes",
    "euphorie", "euphorique", "euphoriques",
    "célébrer", "célèbre", "célèbres", "célèbrent", "célébrait", "célébraient",
    "célébré", "célébrée", "célébrés", "célébrées",
    "fête", "festif", "festive", "festifs", "festives",
    # Émotion générale
    "ému", "émue", "émus", "émues", "émotion", "émotions", "émotionnel", "émotionnelle",
    "émotionnels", "émotionnelles", "émotionnellement",
    "touché", "touchée", "touchés", "touchées",
    "ressentir", "ressent", "ressentais", "ressentait", "ressentaient",
    "ressenti", "ressentie", "ressentis", "ressenties",
    # Tristesse
    "triste", "tristes", "tristement", "tristesse",
    "déçu", "déçue", "déçus", "déçues", "déception",
    "décevoir", "déçoit", "décevait", "décevaient",
    "pleurer", "pleure", "pleures", "pleurent", "pleurait", "pleuraient",
    "pleurs", "larmes",
    "abattu", "abattue", "abattus", "abattues", "abattement",
    "désespéré", "désespérée", "désespérés", "désespérées", "désespoir",
    "malheureux", "malheureuse", "malheureux", "malheureuses", "malheur",
    # Stress / pression émotionnelle
    "stressé", "stressée", "stressés", "stressées", "stress",
    "nerveux", "nerveuse", "nerveux", "nerveuses", "nerveusement", "nervosité",
    "anxieux", "anxieuse", "anxieux", "anxieuses", "anxiété",
    "peur", "effrayé", "effrayée", "effrayés", "effrayées",
    "craindre", "craint", "craignait", "craignaient",
]

# Attention, risque de confusion: peut ça peut être pour un saut. D'où le robustness check avec cosine/embeddings
apparence_physique = [
    # Visage
    "visage", "visages", "face", "faces",
    "yeux", "oeil", "œil", "regard", "regards",
    "sourire", "sourit", "souriait", "souriaient", "souriant", "souriante",
    "souriants", "souriantes",
    "cheveux", "chevelure", "coiffure", "coiffé", "coiffée", "coiffés", "coiffées",
    "blond", "blonde", "blonds", "blondes",
    "brun", "brune", "bruns", "brunes",
    "roux", "rousse", "roux", "rousses",
    # Corps / silhouette
    "corps", "silhouette", "silhouettes",
    "grand", "grande", "grands", "grandes", "grandeur",
    "petit", "petite", "petits", "petites", "petitesse",
    "mince", "minces", "minceur",
    "svelte", "sveltes",
    "élancé", "élancée", "élancés", "élancées",
    "corpulent", "corpulente", "corpulents", "corpulentes",
    "taille", "tailles",
    "poids",
    # Esthétique / beauté
    "beau", "belle", "beaux", "belles", "beauté",
    "joli", "jolie", "jolis", "jolies", "joliment",
    "mignon", "mignonne", "mignons", "mignonnes",
    "séduisant", "séduisante", "séduisants", "séduisantes", "séduction",
    "élégant", "élégante", "élégants", "élégantes", "élégamment", "élégance",
    "gracieux", "gracieuse", "gracieux", "gracieuses", "gracieusement", "grâce",
    "ravissant", "ravissante", "ravissants", "ravissantes",
    "attrayant", "attrayante", "attrayants", "attrayantes",
    "charme", "charmant", "charmante", "charmants", "charmantes",
    "physiquement",
    # Tenue / vêtements
    "tenue", "tenues",
    "vêtement", "vêtements",
    "maillot", "maillots",
    "équipement", "équipements",
]

vie_personnelle = [
    # Famille
    "famille", "familial", "familiale", "familiaux", "familiales",
    "père", "pères", "mère", "mères", "parent", "parents",
    "mari", "maris", "femme", "femmes", "époux", "épouse", "épouses",
    "enfant", "enfants", "fils", "fille", "filles",
    "frère", "frères", "sœur", "sœurs",
    "grand-père", "grand-mère", "grands-parents",
    "bébé", "bébés", "grossesse", "enceinte",
    "conjoint", "conjointe", "conjoints", "conjointes",
    "compagnon", "compagne", "compagnons", "compagnes",
    "couple", "couples",
    # Vie privée
    "vie", "privé", "privée", "privés", "privées",
    "personnel", "personnelle", "personnels", "personnelles",
    "intime", "intimes", "intimité",
    "maison", "maisons", "domicile", "domiciles",
    "quotidien", "quotidienne", "quotidiens", "quotidiennes",
    # Passé / parcours
    "passé", "histoire", "histoires",
    "carrière", "carrières",
    "débuter", "débute", "débutait", "débutaient",
    "débuté", "débutée", "débutés", "débutées",
    "jeunesse", "jeune", "jeunes",
    "enfance", "adolescence",
    "origine", "origines", "natif", "native", "natifs", "natives",
    "né", "née", "nés", "nées", "naissance",
    "grandir", "grandit", "grandissait", "grandissaient",
    # Encadrement / entraîneur
    "entraîneur", "entraîneure", "entraîneurs", "entraîneures",
    "coach", "coachs", "coacher", "coache", "coachait", "coachaient",
    "staff", "équipe technique",
    "préparateur", "préparatrice", "préparateurs", "préparatrices",
    "mentor", "mentors",
    # Blessures / santé personnelle
    "blessure", "blessures", "blessé", "blessée", "blessés", "blessées",
    "opération", "opérations", "opéré", "opérée", "opérés", "opérées",
    "rééducation", "convalescence",
    "santé", "maladie", "maladies", "malade", "malades",
]

personnalite = [
    # Humilité
    "humble", "humbles", "humblement", "humilité",
    "modeste", "modestes", "modestement", "modestie",
    "discret", "discrète", "discrets", "discrètes", "discrètement", "discrétion",
    "effacé", "effacée", "effacés", "effacées",
    # Générosité / altruisme
    "généreux", "généreuse", "généreux", "généreuses", "généreusement", "générosité",
    "altruiste", "altruistes", "altruisme",
    "solidaire", "solidaires", "solidarité",
    "bienveillant", "bienveillante", "bienveillants", "bienveillantes", "bienveillance",
    # Sympathie / charisme
    "sympathique", "sympathiques", "sympathiquement", "sympathie",
    "charismatique", "charismatiques", "charisme",
    "attachant", "attachante", "attachants", "attachantes",
    "populaire", "populaires", "popularité",
    "apprécié", "appréciée", "appréciés", "appréciées",
    "aimé", "aimée", "aimés", "aimées",
    # Sérieux / professionnalisme
    "sérieux", "sérieuse", "sérieux", "sérieuses", "sérieusement", "sérieux",
    "professionnel", "professionnelle", "professionnels", "professionnelles", "professionnellement",
    "rigoureux", "rigoureuse", "rigoureux", "rigoureuses", "rigoureusement", "rigueur",
    "appliqué", "appliquée", "appliqués", "appliquées",
    # Caractère / tempérament
    "caractère", "caractères",
    "tempérament", "tempéraments",
    "personnalité", "personnalités",
    "nature", "naturel", "naturelle", "naturels", "naturelles", "naturellement",
    "authentique", "authentiques", "authenticité",
    "sincère", "sincères", "sincèrement", "sincérité",
    # Arrogance / confiance en soi
    "arrogant", "arrogante", "arrogants", "arrogantes", "arrogance",
    "prétentieux", "prétentieuse", "prétentieux", "prétentieuses", "prétention",
    "confiant", "confiante", "confiants", "confiantes", "confiance",
    "fier", "fière", "fiers", "fières", "fierté",
    "orgueilleux", "orgueilleuse", "orgueilleux", "orgueilleuses", "orgueil",
]

chance = [
    # Chance / malchance
    "chance", "chances",
    "chanceux", "chanceuse", "chanceux", "chanceuses", "chanceusement",
    "malchanceux", "malchanceuse", "malchanceux", "malchanceuses", "malchance",
    "veinard", "veinarde", "veinards", "veinards", "veine",
    "déveine",
    # Hasard
    "hasard", "hasards",
    "aléatoire", "aléatoires", "aléatoirement",
    "fortuit", "fortuite", "fortuits", "fortuites", "fortuitement",
    "imprévu", "imprévue", "imprévus", "imprévues",
    "inattendu", "inattendue", "inattendus", "inattendues", "inattendument",
    # Destin / sort
    "destin", "destins", "destinée", "destinées",
    "sort", "sorts",
    "karma",
    "fortune", "fortunes",
    # Opportunité
    "opportunité", "opportunités",
    "occasion", "occasions",
    "aubaine", "aubaines",
    # Verbes
    "avoir de la chance", "avoir la chance",
    "profiter", "profite", "profites", "profitent", "profitait", "profitaient",
    "profité", "profitée", "profités", "profitées",
    "tomber", "tombe", "tombait", "tombaient",
    "bénéficier", "bénéficie", "bénéficiait", "bénéficiaient",
    "bénéficié",
]

performance_heroisme = [
    # Performance / excellence
    "performance", "performances",
    "performant", "performante", "performants", "performantes",
    "performer", "performe", "performes", "performent", "performait", "performaient",
    "excellent", "excellente", "excellents", "excellentes", "excellemment", "excellence",
    "exceller", "excelle", "excelles", "excellent", "excellait", "excellaient",
    "exceptionnel", "exceptionnelle", "exceptionnels", "exceptionnelles", "exceptionnellement",
    "remarquable", "remarquables", "remarquablement",
    "impressionnant", "impressionnante", "impressionnants", "impressionnantes",
    "impressionner", "impressionne", "impressionnait", "impressionnaient",
    # Meilleur / record
    "meilleur", "meilleure", "meilleurs", "meilleures",
    "record", "records",
    "historique", "historiques", "historiquement",
    "inédit", "inédite", "inédits", "inédites",
    "sans précédent",
    "jamais vu",
    "première fois",
    # Héroïsme / légende
    "héros", "héroïne", "héroïnes",
    "héroïque", "héroïques", "héroïquement", "héroïsme",
    "légendaire", "légendaires",
    "légende", "légendes",
    "mythique", "mythiques",
    "grandiose", "grandioses",
    "épique", "épiques",
    "exploit", "exploits",
    "exploit", "exploiter", "exploite", "exploitait", "exploitaient",
    "réaliser", "réalise", "réalisait", "réalisaient",
    "réalisé", "réalisée", "réalisés", "réalisées", "réalisation", "réalisations",
    "accomplissement", "accomplissements",
    "accomplir", "accomplit", "accomplissait", "accomplissaient",
    "accompli", "accomplie", "accomplis", "accomplies",
    # Gloire / consécration
    "gloire", "glorieux", "glorieuse", "glorieux", "glorieuses", "glorieusement",
    "triomphe", "triomphes", "triomphant", "triomphante", "triomphants", "triomphantes",
    "triompher", "triomphe", "triomphait", "triomphaient",
    "consacré", "consacrée", "consacrés", "consacrées", "consécration",
    "sacre", "sacré", "sacrée", "sacrés", "sacrées",
    "champion", "championne", "champions", "championnes", "championnat",
    "titre", "titres",
    "palmarès",
    "podium", "podiums",
]

resultat_enjeux = [
    # Victoire / défaite
    "victoire", "victoires",
    "vaincre", "vainc", "vainquait", "vainquaient",
    "vaincu", "vaincue", "vaincus", "vaincues", "vainqueur", "vainqueurs",
    "défaite", "défaites",
    "perdre", "perd", "perdait", "perdaient",
    "perdu", "perdue", "perdus", "perdues",
    "gagner", "gagne", "gagnes", "gagnent", "gagnait", "gagnaient",
    "gagné", "gagnée", "gagnés", "gagnées",
    "battre", "bat", "battait", "battaient",
    "battu", "battue", "battus", "battues",
    "remporter", "remporte", "remportait", "remportaient",
    "remporté", "remportée", "remportés", "remportées",
    # Score / résultat
    "score", "scores",
    "résultat", "résultats",
    "point", "points",
    "but", "buts",
    "marquer", "marque", "marques", "marquent", "marquait", "marquaient",
    "marqué", "marquée", "marqués", "marquées",
    "mener", "mène", "menait", "menaient",
    "égaliser", "égalise", "égalisait", "égalisaient",
    "égalisé", "égalisée", "égalisés", "égalisées", "égalisation",
    "avantage", "avantages",
    "retard", "retards",
    # Qualification / élimination
    "qualification", "qualifications",
    "qualifier", "qualifie", "qualifiait", "qualifiaient",
    "qualifié", "qualifiée", "qualifiés", "qualifiées",
    "élimination", "éliminations",
    "éliminer", "élimine", "éliminait", "éliminaient",
    "éliminé", "éliminée", "éliminés", "éliminées",
    "passer", "passe", "passait", "passaient",
    # Enjeux / compétition
    "enjeu", "enjeux",
    "finale", "finales",
    "demi-finale", "demi-finales",
    "quart", "quarts",
    "médaille", "médailles",
    "or", "argent", "bronze",
    "podium", "podiums",
    "classement", "classements",
    "classé", "classée", "classés", "classées",
    "rang", "rangs",
    "place", "places",
    "titre", "titres",
    "trophée", "trophées",
    "olympique", "olympiques",
    "jeux", "compétition", "compétitions",
    "tournoi", "tournois",
    "épreuve", "épreuves",
    "match", "matchs", "matches",
    "rencontre", "rencontres",
]

age = [
    # Jeunesse
    "jeune", "jeunes", "jeunesse",
    "junior", "juniors",
    "cadet", "cadette", "cadets", "cadettes",
    "espoir", "espoirs",
    "prometteur", "prometteuse", "prometteurs", "prometteuses", "promesse", "promesses",
    "talent", "talents", "talentueux", "talentueuse", "talentueux", "talentueuses",
    "précoce", "précoces", "précocité",
    "débutant", "débutante", "débutants", "débutantes",
    "novice", "novices",
    # Expérience / vétéran
    "expérimenté", "expérimentée", "expérimentés", "expérimentées", "expérience", "expériences",
    "vétéran", "vétérane", "vétérans", "vétéranes",
    "senior", "seniors",
    "ancien", "ancienne", "anciens", "anciennes",
    "chevronné", "chevronnée", "chevronnés", "chevronnées",
    "aguerri", "aguerrie", "aguerris", "aguerries",
    "sage", "sages", "sagesse",
    "mûr", "mûre", "mûrs", "mûres", "maturité",
    # Âge explicite
    "âge", "âges",
    "ans", "année", "années",
    "vieux", "vieille", "vieux", "vieilles", "vieillesse",
    "carrière longue", "longévité",
    "retraite", "retraité", "retraitée", "retraités", "retraitées",
    "fin de carrière",
]
marqueur_genre = [
    # Féminin explicite
    "féminin", "féminine", "féminins", "féminines", "féminité",
    "femme", "femmes",
    "dame", "dames",
    "fille", "filles",
    "mademoiselle", "demoiselle", "demoiselles",
    "madame", "mesdames",
    # Masculin explicite
    "masculin", "masculine", "masculins", "masculines", "masculinité",
    "homme", "hommes",
    "monsieur", "messieurs",
    "garçon", "garçons",
    "gars",
    # Désignations genrées sportives
    "comme une femme", "comme un homme",
    "féminin", "masculin",
]

violence = [
    # Violence physique
    "violent", "violente", "violents", "violentes", "violemment", "violence",
    "agressif", "agressive", "agressifs", "agressives", "agressivement", "agressivité",
    "agresser", "agresse", "agressait", "agressaient",
    "agressé", "agressée", "agressés", "agressées",
    "brutaliser", "brutalise", "brutalisait", "brutalisaient",
    "brutalisé", "brutalisée", "brutalisés", "brutalisées",
    "brutal", "brutale", "brutaux", "brutales", "brutalement", "brutalité",
    "frapper", "frappe", "frappait", "frappaient",
    "frappé", "frappée", "frappés", "frappées",
    "bousculer", "bouscule", "bousculait", "bousculaient",
    "bousculé", "bousculée", "bousculés", "bousculées",
    "percuter", "percute", "percutait", "percutaient",
    "percuté", "percutée", "percutés", "percutées",
    "collision", "collisions",
    "choc", "chocs",
    "impact", "impacts",
    "contact", "contacts",
    # Intimidation
    "intimider", "intimide", "intimidait", "intimidaient",
    "intimidé", "intimidée", "intimidés", "intimidées", "intimidation",
    "provoquer", "provoque", "provoquait", "provoquaient",
    "provoqué", "provoquée", "provoqués", "provoquées", "provocation", "provocations",
    "menacer", "menace", "menaçait", "menaçaient",
    "menacé", "menacée", "menacés", "menacées", "menace", "menaces",
]

cooperation_equipe = [
    # Coopération / entraide
    "coopération", "coopérer", "coopère", "coopérait", "coopéraient",
    "coopéré",
    "collaborer", "collabore", "collaborait", "collaboraient",
    "collaboré", "collaboration", "collaborations",
    "entraide", "s'entraider", "s'entraide", "s'entraidait", "s'entraidaient",
    "soutenir", "soutient", "soutenait", "soutenaient",
    "soutenu", "soutenue", "soutenus", "soutenues", "soutien", "soutiens",
    "aider", "aide", "aidait", "aidaient",
    "aidé", "aidée", "aidés", "aidées",
    "solidaire", "solidaires", "solidarité",
    "ensemble", "collectif", "collectivement",
    "uni", "unie", "unis", "unies", "unité",
    "cohésion",
    # Équipe
    "équipe", "équipes",
    "coéquipier", "coéquipière", "coéquipiers", "coéquipières",
    "partenaire", "partenaires",
    "groupe", "groupes",
    "collectif", "collectifs", "collective", "collectives",
    "team", "teams",
    "bloc", "blocs",
    # Jeu collectif
    "passer", "passe", "passait", "passaient",
    "passé", "passée", "passés", "passées",
    "transmission", "transmissions",
    "relais", "relayer", "relaye", "relayait", "relayaient",
    "relayé", "relayée", "relayés", "relayées",
    "jouer ensemble", "jeu collectif",
    "construire", "construit", "construisait", "construisaient",
    "construit", "construite", "construits", "construites",
    "organiser", "organise", "organisait", "organisaient",
    "organisé", "organisée", "organisés", "organisées", "organisation",
    "synchroniser", "synchronise", "synchronisait", "synchronisaient",
    "synchronisé", "synchronisée", "synchronisés", "synchronisées", "synchronisation",
    "coordination", "coordonner", "coordonne", "coordonnait", "coordonnaient",
    "coordonné", "coordonnée", "coordonnés", "coordonnées",
    # Communication
    "communiquer", "communique", "communiquait", "communiquaient",
    "communiqué",
    "parler", "parle", "parlait", "parlaient",
    "discuter", "discute", "discutait", "discutaient",
    "discuté",
    "signal", "signaux", "signaler", "signale", "signalait", "signalaient",
]

analytique = [
    # Statistiques
    "statistique", "statistiques", "statistiquement",
    "stat", "stats",
    "chiffre", "chiffres",
    "donnée", "données",
    "pourcentage", "pourcentages",
    "ratio", "ratios",
    "moyenne", "moyennes",
    "score", "scores",
    "mesure", "mesures",
    "indicateur", "indicateurs",
    "performance", "performances",
    # Analyse
    "analyser", "analyse", "analyses", "analysait", "analysaient",
    "analysé", "analysée", "analysés", "analysées",
    "étudier", "étudie", "étudiait", "étudiaient",
    "étudié", "étudiée", "étudiés", "étudiées",
    "observer", "observe", "observait", "observaient",
    "observé", "observée", "observés", "observées", "observation", "observations",
    "évaluer", "évalue", "évaluait", "évaluaient",
    "évalué", "évaluée", "évalués", "évaluées", "évaluation", "évaluations",
    "calculer", "calcule", "calculait", "calculaient",
    "calculé", "calculée", "calculés", "calculées", "calcul", "calculs",
    "mesurer", "mesure", "mesurait", "mesuraient",
    "mesuré", "mesurée", "mesurés", "mesurées",
    "comparer", "compare", "comparait", "comparaient",
    "comparé", "comparée", "comparés", "comparées", "comparaison", "comparaisons",
    # Précision / détail
    "précis", "précise", "précisément", "précision", "précisions",
    "détail", "détails", "détaillé", "détaillée", "détaillés", "détaillées",
    "exact", "exacte", "exacts", "exactes", "exactement", "exactitude",
    "rigoureux", "rigoureuse", "rigoureux", "rigoureuses", "rigoureusement", "rigueur",
    "objectif", "objective", "objectifs", "objectives", "objectivement", "objectivité",
    # Références chiffrées
    "temps", "chrono", "chronométrer", "chronomètre",
    "kilomètre", "kilomètres",
    "mètre", "mètres",
    "seconde", "secondes",
    "minute", "minutes",
    "classement", "classements",
    "rang", "rangs",
    "place", "places",
]

encouragement = [
    # Encouragements directs
    "bravo", "félicitations", "féliciter", "félicite", "félicitait", "félicitaient",
    "félicité", "félicitée", "félicités", "félicitées",
    "congratuler", "congratule", "congratulait", "congratulaient",
    "congratulé", "congratulée", "congratulés", "congratulées",
    "applaudir", "applaudit", "applaudissait", "applaudissaient",
    "applaudi", "applaudie", "applaudis", "applaudies",
    "applaudissement", "applaudissements",
    "ovation", "ovations",

    # Interjections / exclamations
    "allez", "vas-y", "go", "hop", "hep",
    "oh", "ah", "oh là là", "waouh", "wow",
    "ouais", "oui oui", "voilà voilà",
    "aïe", "aïe aïe", "aïe aïe aïe",
    "hé", "hey",

    # Expressions positives courtes
    "super", "excellent", "parfait", "génial", "géniale",
    "chapeau", "respect", "classe",
    "bien", "très bien", "c'est bien", "bien joué", "bien fait",
    "beau geste", "beau travail", "beau jeu",
    "magnifique", "superbe", "sublime",
    "incroyable", "extraordinaire", "fantastique", "formidable",
    "impressionnant", "impressionnante", "impressionnants", "impressionnantes",
    "époustouflant", "époustouflante", "époustouflants", "époustouflantes",
    "admirable", "admirables", "admiration",
    "remarquable", "remarquables", "remarquablement",
    "exceptionnel", "exceptionnelle", "exceptionnels", "exceptionnelles",
    "sensationnel", "sensationnelle", "sensationnels", "sensationnelles",
    "formidable", "formidables",
    "fabuleux", "fabuleuse", "fabuleux", "fabuleuses",
    "merveilleux", "merveilleuse", "merveilleux", "merveilleuses",

    # Soutien / encouragement
    "encourager", "encourage", "encourages", "encouragent", "encourageait", "encourageaient",
    "encouragé", "encouragée", "encouragés", "encouragées", "encouragement", "encouragements",
    "soutenir", "soutient", "soutenait", "soutenaient",
    "soutenu", "soutenue", "soutenus", "soutenues", "soutien", "soutiens",
    "supporter", "supporte", "supportait", "supportaient",
    "supporteur", "supporteurs", "supportrice", "supportrices",
    "fan", "fans",
    "public", "publics", "foule", "foules",
    "tribune", "tribunes",

    # Admiration
    "admirer", "admire", "admirait", "admiraient",
    "admiré", "admirée", "admirés", "admirées",

    # Positivité générale
    "positif", "positive", "positifs", "positives", "positivement",
    "optimiste", "optimistes", "optimisme",
    "confiant", "confiante", "confiants", "confiantes", "confiance",
]

strategie = [
    # Stratégie générale
    "stratégie", "stratégies", "stratégique", "stratégiques", "stratégiquement",
    "tactique", "tactiques", "tactiquement",
    "plan", "plans", "planifier", "planifie", "planifiait", "planifiaient",
    "planifié", "planifiée", "planifiés", "planifiées", "planification",
    "schéma", "schémas",
    "système", "systèmes",
    "dispositif", "dispositifs",
    # Décision / choix
    "décider", "décide", "décidait", "décidaient",
    "décidé", "décidée", "décidés", "décidées", "décision", "décisions",
    "choisir", "choisit", "choisissait", "choisissaient",
    "choisi", "choisie", "choisis", "choisies", "choix",
    "opter", "opte", "optait", "optaient",
    "opté",
    "privilégier", "privilégie", "privilégiait", "privilégiaient",
    "privilégié", "privilégiée", "privilégiés", "privilégiées",
    # Anticipation / lecture du jeu
    "anticiper", "anticipe", "anticipait", "anticipaient",
    "anticipé", "anticipée", "anticipés", "anticipées", "anticipation",
    "prévoir", "prévoit", "prévoyait", "prévoyaient",
    "prévu", "prévue", "prévus", "prévues",
    "lire", "lit", "lisait", "lisaient", "lecture",
    "analyser", "analyse", "analysait", "analysaient",
    "analysé", "analysée", "analysés", "analysées",
    "réfléchir", "réfléchit", "réfléchissait", "réfléchissaient",
    "réfléchi", "réfléchie", "réfléchis", "réfléchies",
    # Adaptation
    "adapter", "adapte", "adaptait", "adaptaient",
    "adapté", "adaptée", "adaptés", "adaptées", "adaptation", "adaptations",
    "ajuster", "ajuste", "ajustait", "ajustaient",
    "ajusté", "ajustée", "ajustés", "ajustées", "ajustement", "ajustements",
    "corriger", "corrige", "corrigeait", "corrigeaient",
    "corrigé", "corrigée", "corrigés", "corrigées", "correction", "corrections",
    "modifier", "modifie", "modifiait", "modifiaient",
    "modifié", "modifiée", "modifiés", "modifiées", "modification", "modifications",
    # Prise de risque
    "risque", "risques", "risquer", "risque", "risquait", "risquaient",
    "risqué", "risquée", "risqués", "risquées",
    "audacieux", "audacieuse", "audacieux", "audacieuses", "audace",
    "oser", "ose", "osait", "osaient",
    "osé", "osée", "osés", "osées",
    "prudent", "prudente", "prudents", "prudentes", "prudemment", "prudence",
    "conservateur", "conservatrice", "conservateurs", "conservatrices",
    # Positionnement / placement
    "positionner", "positionne", "positionnait", "positionnaient",
    "positionné", "positionnée", "positionnés", "positionnées", "positionnement",
    "placer", "place", "plaçait", "plaçaient",
    "placé", "placée", "placés", "placées", "placement",
    "couvrir", "couvre", "couvrait", "couvraient",
    "couvert", "couverte", "couverts", "couvertes", "couverture",
]

Première étape : ces topics identifiés dans la littérature sont ils bien présents dans nos données ? 

Pour savoir, on calcule un score par phrase i.e. score_theme_n = nombre_mots_theme_n/nombre_mots_phrase.

In [14]:
#Score
dictionnaires = {
    "forme_physique": forme_physique,
    "force_caractere_effort": force_caractere_effort,
    "mental_leadership": mental_leadership,
    "emotion": emotion,
    "apparence_physique": apparence_physique,
    "vie_personnelle": vie_personnelle,
    "personnalite": personnalite,
    "chance": chance,
    "performance_heroisme": performance_heroisme,
    "resultat_enjeux": resultat_enjeux,
    "age": age,
    "marqueur_genre": marqueur_genre,
    "violence": violence,
    "cooperation_equipe": cooperation_equipe,
    "analytique": analytique,
    "encouragement": encouragement,
    "strategie": strategie,
}

def score_theme(text, dictionary):
    words = text.lower().split()
    matches = sum(1 for word in words if word in dictionary)
    return matches / len(words) if len(words) > 0 else 0

for theme, dico in dictionnaires.items():
    dico_set = set(dico)  # conversion en set pour la rapidité
    df[theme] = df["text"].apply(lambda x: score_theme(x, dico_set))

df.head()

,start,stop,text,main_speaker,main_g,audio_file,sport,ID,g_ath,forme_physique,...,chance,performance_heroisme,resultat_enjeux,age,marqueur_genre,violence,cooperation_equipe,analytique,encouragement,strategie
1,123.44,139.68,et pour une histoire racontée comme si on ét...,SPEAKER_03,female,PD2_wav.wav,patin_art,PD2,M,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
1,123.44,139.68,choix musical de c'est vraiment pour créer ce...,SPEAKER_03,female,PD2_wav.wav,patin_art,PD2,M,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.100000
1,123.44,139.68,sais qu'il fait partie d'un de vos programmes...,SPEAKER_03,female,PD2_wav.wav,patin_art,PD2,M,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
2,139.88,189.72,'est pour moi une des plus belles chorégraphie...,SPEAKER_04,female,PD2_wav.wav,patin_art,PD2,M,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
2,139.88,189.72,pourquoi déjà le choix musical est très très b...,SPEAKER_04,female,PD2_wav.wav,patin_art,PD2,M,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.071429,0.071429


On regarde la part de phrases concernées par chaque thème ie contenant au moins un mot du dictionnaire du thème en question.

In [ ]:
themes = list(dictionnaires.keys())
sports = df["sport"].unique()

rows = []
for theme in themes:
    row = {"theme": theme}
    
    # Global
    n = (df[theme] > 0).sum()
    row["global_%"] = round(n / len(df) * 100, 1)
    
    # Par sport
    for sport in sports:
        df_sport = df[df["sport"] == sport]
        n_sport = (df_sport[theme] > 0).sum()
        row[f"{sport}_%"] = round(n_sport / len(df_sport) * 100, 1)
    
    rows.append(row)

freq_df = pd.DataFrame(rows).set_index("theme")
print(freq_df.to_string())

                        global_%  patin_art_%  biathlon_%  ski_alpin_%  ski_free_%  curling_%  patin_vit_%
theme                                                                                                     
forme_physique               5.2          5.8         7.1          6.1         2.9        3.4          7.8
force_caractere_effort       3.1          4.1         2.8          1.7         2.6        2.6          3.7
mental_leadership            2.0          1.1         1.9          1.2         1.3        3.5          0.8
emotion                      1.3          2.4         1.1          1.2         1.3        0.8          1.4
apparence_physique           8.6          9.0         8.1          6.7        10.6        7.3          9.0
vie_personnelle              4.7          9.0         3.6          1.6         3.4        3.7          4.1
personnalite                 0.8          1.8         0.5          0.8         0.6        0.6          0.3
chance                       1.0     

Test de significativité : Kruskal-Wallis
Equivalent non-paramétrique de l'ANOVA i.e. pas d'hypothèse de normalité, adapté à ces scores avec beaucoup de zéros.

In [19]:
!pip install scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 95.2 MB/s  0:00:00m eta 0:00:01


In [20]:
from scipy.stats import kruskal

for theme in themes:
    groupes = [df[df["sport"] == sport][theme].values for sport in sports]
    stat, pval = kruskal(*groupes)
    print(f"{theme}: p={pval:.4f} {'Significatif seuil 0.05' if pval < 0.05 else ''}")

forme_physique: p=0.0000 Significatif seuil 0.05
force_caractere_effort: p=0.0000 Significatif seuil 0.05
mental_leadership: p=0.0000 Significatif seuil 0.05
emotion: p=0.0000 Significatif seuil 0.05
apparence_physique: p=0.0000 Significatif seuil 0.05
vie_personnelle: p=0.0000 Significatif seuil 0.05
personnalite: p=0.0000 Significatif seuil 0.05
chance: p=0.0000 Significatif seuil 0.05
performance_heroisme: p=0.0000 Significatif seuil 0.05
resultat_enjeux: p=0.0000 Significatif seuil 0.05
age: p=0.0000 Significatif seuil 0.05
marqueur_genre: p=0.0000 Significatif seuil 0.05
violence: p=0.0000 Significatif seuil 0.05
cooperation_equipe: p=0.0000 Significatif seuil 0.05
analytique: p=0.0000 Significatif seuil 0.05
encouragement: p=0.0000 Significatif seuil 0.05
strategie: p=0.0000 Significatif seuil 0.05


Visualisation.

In [21]:
!pip install statsmodels --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 93.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [statsmodels] [statsmodels]


In [23]:
!pip install matplotlib

  Using cached matplotlib-3.10.9-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 67.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 91.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 29.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [matplotlib]6 [matplotlib]


In [26]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.stats.proportion import proportion_confint

sports = df["sport"].unique()
themes = list(dictionnaires.keys())

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=True)
axes = axes.flatten()

for ax, sport in zip(axes, sports):
    df_sport = df[df["sport"] == sport]
    n = len(df_sport)
    
    frequences, ci_low, ci_high = [], [], []
    
    for theme in themes:
        p = (df_sport[theme] > 0).sum() / n
        ci = proportion_confint(
            count=(df_sport[theme] > 0).sum(),
            nobs=n,
            alpha=0.05,
            method="wilson"
        )
        frequences.append(p * 100)
        ci_low.append((p - ci[0]) * 100)
        ci_high.append((ci[1] - p) * 100)
    
    x = np.arange(len(themes))
    ax.bar(x, frequences, alpha=0.8, color="steelblue",
           yerr=[ci_low, ci_high], capsize=3, error_kw={"linewidth": 0.8})
    ax.set_xticks(x)
    ax.set_xticklabels(themes, rotation=45, ha="right", fontsize=8)
    ax.set_title(sport)
    ax.set_ylabel("% phrases")

fig.suptitle("Fréquence des thèmes par sport (IC 95% Wilson)", fontsize=13)
plt.tight_layout()
plt.savefig("barplot_frequences_par_sport.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sauvegardé")

Sauvegardé


Robustness check : On regarde le pourcentage de phrases **longues** (top quartile) où cette fois-ci **deux** mots d'un dictionnaire sont inclus. Cela évite que dans des phrases longues, un seul mot soit là "par hasard" sans pour autant être central/donner du sens à la prhase en question.

In [27]:
# Calculer le seuil
q75 = df["text"].str.split().str.len().quantile(0.75)
print(f"Seuil top quartile : {q75} mots")

# Filtrer
df_long = df[df["text"].str.split().str.len() >= q75].copy()
print(f"Phrases conservées : {len(df_long)} / {len(df)}")

# Précalculer les sets une seule fois
dico_sets = {theme: set(dictionnaires[theme]) for theme in themes}

# Précalculer les mots de chaque phrase une seule fois
df_long["words"] = df_long["text"].str.lower().str.split()

def count_matches_fast(words, dico_set):
    return sum(1 for word in words if word in dico_set)

rows = []
for theme in themes:
    row = {"theme": theme}
    dico_set = dico_sets[theme]
    
    counts = df_long["words"].apply(lambda w: count_matches_fast(w, dico_set))
    
    # Seuil 1 mot
    n1 = (counts >= 1).sum()
    row["global_1mot_%"] = round(n1 / len(df_long) * 100, 1)
    
    # Seuil 2 mots
    n2 = (counts >= 2).sum()
    row["global_2mots_%"] = round(n2 / len(df_long) * 100, 1)
    
    for sport in sports:
        mask = df_long["sport"] == sport
        n1_sport = (counts[mask] >= 1).sum()
        n2_sport = (counts[mask] >= 2).sum()
        row[f"{sport}_1mot_%"] = round(n1_sport / mask.sum() * 100, 1)
        row[f"{sport}_2mots_%"] = round(n2_sport / mask.sum() * 100, 1)
    
    rows.append(row)

freq_df = pd.DataFrame(rows).set_index("theme")
print(freq_df.to_string())

Seuil top quartile : 13.0 mots
Phrases conservées : 11522 / 41359
                        global_1mot_%  global_2mots_%  patin_art_1mot_%  patin_art_2mots_%  biathlon_1mot_%  biathlon_2mots_%  ski_alpin_1mot_%  ski_alpin_2mots_%  ski_free_1mot_%  ski_free_2mots_%  curling_1mot_%  curling_2mots_%  patin_vit_1mot_%  patin_vit_2mots_%
theme                                                                                                                                                                                                                                                                      
forme_physique                    9.9             1.2              11.1                1.6             13.6               1.5              20.0                2.2              6.4               0.5             5.6              0.6              14.8                2.2
force_caractere_effort            6.6             0.6               8.2                0.8              6.0               0.5     

Ski alpin n'a que 4 fichiers donc très peu de phrases y compris des phrases "longues". Les zéros sont probablement dus à un corpus trop petit pour ce sport, pas à une absence réelle du thème.

Apparence physique : chiffres semblent TRES élevés, on enlève donc des mots potentiellement tendencieux (beau, grand, élancé, *etc*) en guise de robustness check. On refait les mêmes analyses.

In [33]:
apparence_physique_bis = [
    # Visage
    "visage", "visages",
    "yeux", "oeil", "œil",
    "sourire", "sourit", "souriait", "souriaient", "souriant", "souriante",
    "souriants", "souriantes",
    "cheveux", "chevelure", "coiffure", "coiffé", "coiffée", "coiffés", "coiffées",
    "blond", "blonde", "blonds", "blondes",
    "brun", "brune", "bruns", "brunes",
    "roux", "rousse", "rousses",

    # Corps / silhouette
    "corps", "silhouette", "silhouettes",
    "mince", "minces", "minceur",
    "svelte", "sveltes",
    "corpulent", "corpulente", "corpulents", "corpulentes",
    "poids",

    # Esthétique / beauté
    "mignon", "mignonne", "mignons", "mignonnes",
    "séduisant", "séduisante", "séduisants", "séduisantes", "séduction",
    "élégant", "élégante", "élégants", "élégantes", "élégamment", "élégance",
    "gracieux", "gracieuse", "gracieuses", "gracieusement", "grâce",
    "ravissant", "ravissante", "ravissants", "ravissantes",
    "attrayant", "attrayante", "attrayants", "attrayantes",
    "charme", "charmant", "charmante", "charmants", "charmantes",
    "physiquement",

    # Tenue / vêtements
    "vêtement", "vêtements",
    "maillot", "maillots",
]

In [34]:
#Score
dictionnaires_bis = {
    "forme_physique": forme_physique,
    "force_caractere_effort": force_caractere_effort,
    "mental_leadership": mental_leadership,
    "emotion": emotion,
    "apparence_physique": apparence_physique,
    "apparence_physique_bis": apparence_physique_bis,
    "vie_personnelle": vie_personnelle,
    "personnalite": personnalite,
    "chance": chance,
    "performance_heroisme": performance_heroisme,
    "resultat_enjeux": resultat_enjeux,
    "age": age,
    "marqueur_genre": marqueur_genre,
    "violence": violence,
    "cooperation_equipe": cooperation_equipe,
    "analytique": analytique,
    "encouragement": encouragement,
    "strategie": strategie,
}

def score_theme(text, dictionary):
    words = text.lower().split()
    matches = sum(1 for word in words if word in dictionary)
    return matches / len(words) if len(words) > 0 else 0

df_bis = df.copy()

for theme, dico in dictionnaires_bis.items():
    dico_set_bis = set(dico)  # conversion en set pour la rapidité
    df_bis[theme] = df_bis["text"].apply(lambda x: score_theme(x, dico_set_bis))

df_bis.head()

,start,stop,text,main_speaker,main_g,audio_file,sport,ID,g_ath,forme_physique,...,performance_heroisme,resultat_enjeux,age,marqueur_genre,violence,cooperation_equipe,analytique,encouragement,strategie,apparence_physique_bis
1,123.44,139.68,et pour une histoire racontée comme si on ét...,SPEAKER_03,female,PD2_wav.wav,patin_art,PD2,M,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0
1,123.44,139.68,choix musical de c'est vraiment pour créer ce...,SPEAKER_03,female,PD2_wav.wav,patin_art,PD2,M,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.100000,0.0
1,123.44,139.68,sais qu'il fait partie d'un de vos programmes...,SPEAKER_03,female,PD2_wav.wav,patin_art,PD2,M,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0
2,139.88,189.72,'est pour moi une des plus belles chorégraphie...,SPEAKER_04,female,PD2_wav.wav,patin_art,PD2,M,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0
2,139.88,189.72,pourquoi déjà le choix musical est très très b...,SPEAKER_04,female,PD2_wav.wav,patin_art,PD2,M,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.071429,0.071429,0.0


On regarde différence entre version originale de apparence physique et celle plus "pure".

In [35]:
def score_theme(text, dictionary):
    words = text.lower().split()
    matches = sum(1 for word in words if word in dictionary)
    return matches / len(words) if len(words) > 0 else 0

df_test = df.copy()

for theme, dico in dictionnaires_bis.items():
    dico_set_bis = set(dico)  # conversion en set pour la rapidité
    df_bis[theme] = df_bis["text"].apply(lambda x: score_theme(x, dico_set_bis))

df_bis.head()


themes = list(dictionnaires_bis.keys())
sports = df_bis["sport"].unique()

rows = []
for theme in themes:
    row = {"theme": theme}
    
    # Global
    n_bis = (df_bis[theme] > 0).sum()
    row["global_%"] = round(n_bis / len(df_bis) * 100, 1)
    
    # Par sport
    for sport in sports:
        df_sport_bis = df_bis[df_bis["sport"] == sport]
        n_sport_bis = (df_sport_bis[theme] > 0).sum()
        row[f"{sport}_%"] = round(n_sport_bis / len(df_sport_bis) * 100, 1)
    
    rows.append(row)

freq_df_bis = pd.DataFrame(rows).set_index("theme")
print(freq_df_bis.to_string())

                        global_%  patin_art_%  biathlon_%  ski_alpin_%  ski_free_%  curling_%  patin_vit_%
theme                                                                                                     
forme_physique               5.2          5.8         7.1          6.1         2.9        3.4          7.8
force_caractere_effort       3.1          4.1         2.8          1.7         2.6        2.6          3.7
mental_leadership            2.0          1.1         1.9          1.2         1.3        3.5          0.8
emotion                      1.3          2.4         1.1          1.2         1.3        0.8          1.4
apparence_physique           8.6          9.0         8.1          6.7        10.6        7.3          9.0
apparence_physique_bis       1.0          1.7         0.8          0.8         0.6        0.7          1.3
vie_personnelle              4.7          9.0         3.6          1.6         3.4        3.7          4.1
personnalite                 0.8     

On voit qu'*a priori* l'omniprésence de l'apparence physique (contradictoire avec la littérature) était tirée par des termes ambigus comme "beau" ou "grand".

In [36]:
dictionnaires_fin = {
    "forme_physique": forme_physique,
    "force_caractere_effort": force_caractere_effort,
    "mental_leadership": mental_leadership,
    "emotion": emotion,
    "apparence_physique": apparence_physique_bis,
    "vie_personnelle": vie_personnelle,
    "personnalite": personnalite,
    "chance": chance,
    "performance_heroisme": performance_heroisme,
    "resultat_enjeux": resultat_enjeux,
    "age": age,
    "marqueur_genre": marqueur_genre,
    "violence": violence,
    "cooperation_equipe": cooperation_equipe,
    "analytique": analytique,
    "encouragement": encouragement,
    "strategie": strategie,
}

themes = list(dictionnaires_fin.keys())
sports = df["sport"].unique()
for theme, dico in dictionnaires_fin.items():
    dico_set = set(dico)
    df[theme] = df["text"].apply(lambda x: sum(1 for w in str(x).lower().split() if w in dico_set) / max(len(str(x).split()), 1))
rows = []
for theme in themes:
    row = {"theme": theme}
    
    # Global
    n = (df[theme] > 0).sum()
    row["global_%"] = round(n / len(df) * 100, 1)
    
    # Par sport
    for sport in sports:
        df_sport = df[df["sport"] == sport]
        n_sport = (df_sport[theme] > 0).sum()
        row[f"{sport}_%"] = round(n_sport / len(df_sport) * 100, 1)
    
    rows.append(row)

freq_df = pd.DataFrame(rows).set_index("theme")
print(freq_df.to_string())

                        global_%  patin_art_%  biathlon_%  ski_alpin_%  ski_free_%  curling_%  patin_vit_%
theme                                                                                                     
forme_physique               5.2          5.8         7.1          6.1         2.9        3.4          7.8
force_caractere_effort       3.1          4.1         2.8          1.7         2.6        2.6          3.7
mental_leadership            2.0          1.1         1.9          1.2         1.3        3.5          0.8
emotion                      1.3          2.4         1.1          1.2         1.3        0.8          1.4
apparence_physique           1.0          1.7         0.8          0.8         0.6        0.7          1.3
vie_personnelle              4.7          9.0         3.6          1.6         3.4        3.7          4.1
personnalite                 0.8          1.8         0.5          0.8         0.6        0.6          0.3
chance                       1.0     

In [37]:
# significativité

from scipy.stats import kruskal

for theme in themes:
    groupes = [df[df["sport"] == sport][theme].values for sport in sports]
    stat, pval = kruskal(*groupes)
    print(f"{theme}: p={pval:.4f} {'Significatif seuil 0.05' if pval < 0.05 else ''}")

forme_physique: p=0.0000 Significatif seuil 0.05
force_caractere_effort: p=0.0000 Significatif seuil 0.05
mental_leadership: p=0.0000 Significatif seuil 0.05
emotion: p=0.0000 Significatif seuil 0.05
apparence_physique: p=0.0000 Significatif seuil 0.05
vie_personnelle: p=0.0000 Significatif seuil 0.05
personnalite: p=0.0000 Significatif seuil 0.05
chance: p=0.0000 Significatif seuil 0.05
performance_heroisme: p=0.0000 Significatif seuil 0.05
resultat_enjeux: p=0.0000 Significatif seuil 0.05
age: p=0.0000 Significatif seuil 0.05
marqueur_genre: p=0.0000 Significatif seuil 0.05
violence: p=0.0000 Significatif seuil 0.05
cooperation_equipe: p=0.0000 Significatif seuil 0.05
analytique: p=0.0000 Significatif seuil 0.05
encouragement: p=0.0000 Significatif seuil 0.05
strategie: p=0.0000 Significatif seuil 0.05


In [40]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.stats.proportion import proportion_confint

sports = df["sport"].unique()
themes = list(dictionnaires_fin.keys())

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=True)
axes = axes.flatten()

for ax, sport in zip(axes, sports):
    df_sport = df[df["sport"] == sport]
    n = len(df_sport)
    
    frequences, ci_low, ci_high = [], [], []
    
    for theme in themes:
        p = (df_sport[theme] > 0).sum() / n
        ci = proportion_confint(
            count=(df_sport[theme] > 0).sum(),
            nobs=n,
            alpha=0.05,
            method="wilson"
        )
        frequences.append(p * 100)
        ci_low.append((p - ci[0]) * 100)
        ci_high.append((ci[1] - p) * 100)
    
    x = np.arange(len(themes))
    ax.bar(x, frequences, alpha=0.8, color="steelblue",
           yerr=[ci_low, ci_high], capsize=3, error_kw={"linewidth": 0.8})
    ax.set_xticks(x)
    ax.set_xticklabels(themes, rotation=45, ha="right", fontsize=8)
    ax.set_title(sport)
    ax.set_ylabel("% phrases")

fig.suptitle("Fréquence des thèmes par sport (IC 95% Wilson)", fontsize=13)
plt.tight_layout()
plt.savefig("barplot_frequences_par_sport_final.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sauvegardé")

Sauvegardé


Le robustness check a permis de constater le problème : le dictionnaire **apparence physique** était boosté par des termes ambigus. On a refait les stats sans ces terms ambigus. L'omniprésence constatée était suspecte au regard de la littérature, contrairement à resultat_enjeux qu'il est "logique" de retrouver massivement dans les commentaires.

**Maintenant on regarde par genre (athlètes et journalistes) et par sport.**

In [43]:
import scipy.stats as stats

themes = list(dictionnaires_fin.keys())
sports = df["sport"].unique()

for analyse, genre_col, label_H, label_F in [
    ("g_ath", "g_ath", "H", "F"),
    ("main_g", "main_g", "male", "female")
]:
    print(f"{'ATHLETES (H1)' if genre_col == 'g_ath' else 'JOURNALISTS (H2)'}")
    
    for sport in sports:
        print(f"\n {sport} ")
        df_sport = df[df["sport"] == sport]
        
        resultats = []
        for theme in themes:
            groupe_H = df_sport[df_sport[genre_col] == label_H][theme]
            groupe_F = df_sport[df_sport[genre_col] == label_F][theme]
            
            if len(groupe_H) == 0 or len(groupe_F) == 0:
                continue
            
            stat, pval = stats.mannwhitneyu(groupe_H, groupe_F, alternative="two-sided")
            
            resultats.append({
                "theme": theme,
                "mean_H": groupe_H.mean(),
                "mean_F": groupe_F.mean(),
                "diff": groupe_F.mean() - groupe_H.mean(),
                "p_value": pval,
                "significatif": "*" if pval < 0.05 else ""
            })
        
        resultats_df = pd.DataFrame(resultats).sort_values("p_value")
        print(resultats_df.to_string())

ATHLETES (H1)

 patin_art 
                     theme    mean_H    mean_F      diff   p_value significatif
11          marqueur_genre  0.000969  0.002342  0.001374  0.000501            *
10                     age  0.003006  0.005272  0.002266  0.002707            *
15           encouragement  0.016948  0.021024  0.004076  0.007140            *
6             personnalite  0.000936  0.002436  0.001500  0.021759            *
5          vie_personnelle  0.005973  0.006939  0.000966  0.030981            *
4       apparence_physique  0.001427  0.002080  0.000654  0.032839            *
12                violence  0.000045  0.000000 -0.000045  0.051016             
2        mental_leadership  0.001527  0.000575 -0.000952  0.085476             
14              analytique  0.007510  0.009334  0.001824  0.105023             
16               strategie  0.005072  0.004816 -0.000256  0.171104             
1   force_caractere_effort  0.003596  0.004820  0.001225  0.175632             
8     perform

In [44]:
# radar plot

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats

themes = list(dictionnaires_fin.keys())
sports = list(df["sport"].unique())
N = len(themes)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # fermer le polygone

for genre_col, label_H, label_F, titre in [
    ("g_ath", "H", "F", "Athletes (H1)"),
    ("main_g", "male", "female", "Journalists (H2)")
]:
    fig, axes = plt.subplots(1, len(sports), figsize=(6 * len(sports), 6),
                              subplot_kw=dict(polar=True))

    for ax, sport in zip(axes, sports):
        df_sport = df[df["sport"] == sport]

        means_H, means_F, sig = [], [], []
        for theme in themes:
            g_H = df_sport[df_sport[genre_col] == label_H][theme]
            g_F = df_sport[df_sport[genre_col] == label_F][theme]
            means_H.append(g_H.mean())
            means_F.append(g_F.mean())
            _, pval = stats.mannwhitneyu(g_H, g_F, alternative="two-sided")
            sig.append(pval < 0.05)

        # Fermer le polygone
        means_H += means_H[:1]
        means_F += means_F[:1]

        ax.plot(angles, means_H, color="steelblue", linewidth=1.5, label="H/male")
        ax.fill(angles, means_H, color="steelblue", alpha=0.2)
        ax.plot(angles, means_F, color="salmon", linewidth=1.5, label="F/female")
        ax.fill(angles, means_F, color="salmon", alpha=0.2)

        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(themes, size=7)
        ax.set_title(sport, pad=15)
        ax.legend(loc="upper right", bbox_to_anchor=(1.2, 1.1), fontsize=8)

    fig.suptitle(f"Radar — {titre}", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"radar_{genre_col}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Sauvegardé : radar_{genre_col}.png")

Sauvegardé : radar_g_ath.png
Sauvegardé : radar_main_g.png


In [48]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats

themes = list(dictionnaires_fin.keys())
sports = list(df["sport"].unique())
all_groups = ["global"] + sports

def mean_ci(groupe_H, groupe_F, n_boot=500):
    diff = groupe_H.mean() - groupe_F.mean()
    diffs_boot = []
    for _ in range(n_boot):
        boot_H = groupe_H.sample(len(groupe_H), replace=True)
        boot_F = groupe_F.sample(len(groupe_F), replace=True)
        diffs_boot.append(boot_H.mean() - boot_F.mean())
    ci_low = diff - np.percentile(diffs_boot, 2.5)
    ci_high = np.percentile(diffs_boot, 97.5) - diff
    return diff, ci_low, ci_high

for group in all_groups:
    df_group = df if group == "global" else df[df["sport"] == group]

    diffs_ath, ci_low_ath, ci_high_ath = [], [], []
    diffs_jour, ci_low_jour, ci_high_jour = [], [], []

    for theme in themes:
        g_H = df_group[df_group["g_ath"] == "H"][theme]
        g_F = df_group[df_group["g_ath"] == "F"][theme]
        d, cl, ch = mean_ci(g_H, g_F)
        diffs_ath.append(d)
        ci_low_ath.append(cl)
        ci_high_ath.append(ch)

        g_H = df_group[df_group["main_g"] == "male"][theme]
        g_F = df_group[df_group["main_g"] == "female"][theme]
        d, cl, ch = mean_ci(g_H, g_F)
        diffs_jour.append(d)
        ci_low_jour.append(cl)
        ci_high_jour.append(ch)

    x = np.arange(len(themes))
    fig, ax = plt.subplots(figsize=(14, 6))

    ax.bar(x - 0.22, diffs_ath, width=0.38, alpha=0.6, label='Athlètes', color='green')
    ax.bar(x + 0.22, diffs_jour, width=0.38, alpha=0.6, label='Commentateurs', color='orange')
    ax.errorbar(x - 0.22, diffs_ath, yerr=[ci_low_ath, ci_high_ath],
                fmt='none', color='black', capsize=4, linewidth=1.2)
    ax.errorbar(x + 0.22, diffs_jour, yerr=[ci_low_jour, ci_high_jour],
                fmt='none', color='black', capsize=4, linewidth=1.2)

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(themes, rotation=-45, ha="left", fontsize=9)
    ax.set_title(group)
    ax.legend()
    ax.set_ylabel("Différence de genre (H - F)")
    ax.annotate("↑ hommes", xy=(-0.05, 0.92), xycoords='axes fraction', fontsize=9)
    ax.annotate("↓ femmes", xy=(-0.05, 0.05), xycoords='axes fraction', fontsize=9)

    fig.suptitle(f"Différences de genre par thème — {group} (IC 95% bootstrap)", fontsize=12)
    plt.tight_layout()
    plt.savefig(f"barplot_genre_{group}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Sauvegardé : barplot_genre_{group}.png")

Sauvegardé : barplot_genre_global.png
Sauvegardé : barplot_genre_patin_art.png
Sauvegardé : barplot_genre_biathlon.png
Sauvegardé : barplot_genre_ski_alpin.png
Sauvegardé : barplot_genre_ski_free.png
Sauvegardé : barplot_genre_curling.png
Sauvegardé : barplot_genre_patin_vit.png


Etape suivante : **robustness check final**.

Pour vérifier la robustesse des résultats obtenus avec cette analyse par prévalence, on la complète avec une analyse (simple) par similarité cosine. En effet limite de l'approche par prévalence : elle ne capture pas le contexte sémantique dans lequel ces mots sont utilisés. Deux groupes peuvent utiliser les mêmes mots avec une fréquence similaire mais dans des contextes sémantiquement différents.

Principe du robustness check : nous définissons pour chaque thème une phrase-ancre représentative de son sens central. À partir des embeddings CamemBERT, nous calculons ensuite la similarité cosine entre chaque phrase du corpus et cette phrase-ancre. Une similarité cosine moyenne plus élevée pour un groupe de genre indique que le commentaire de ce groupe est sémantiquement plus proche du thème considéré, indépendamment de la présence exacte des mots du dictionnaire. La comparaison de ces distributions entre groupes de genre via des tests de Mann-Whitney permet d'évaluer si les différences thématiques identifiées par l'analyse de prévalence sont robustes.


In [49]:
phrases_ancres = {
    "forme_physique": "cet athlète est très rapide et puissant physiquement",
    "force_caractere_effort": "il se bat avec beaucoup de courage et de détermination",
    "mental_leadership": "elle dirige son équipe avec calme et intelligence tactique",
    "emotion": "quelle joie incroyable, il est ému aux larmes",
    "apparence_physique": "elle est élégante et gracieuse dans ses mouvements",
    "vie_personnelle": "il parle de sa famille et de son enfance",
    "personnalite": "c'est quelqu'un d'humble et de discret",
    "chance": "il a eu beaucoup de chance aujourd'hui",
    "performance_heroisme": "une performance légendaire, un exploit historique",
    "resultat_enjeux": "victoire finale, elle remporte la médaille d'or",
    "age": "ce jeune athlète prometteur débute sa carrière",
    "marqueur_genre": "la joueuse féminine affronte l'équipe masculine",
    "violence": "une faute brutale et agressive",
    "cooperation_equipe": "l'équipe joue ensemble avec une belle solidarité",
    "analytique": "les statistiques montrent une précision remarquable",
    "encouragement": "bravo, c'est magnifique, continuez comme ça",
    "strategie": "une stratégie tactique bien pensée et anticipée",
}

Les phrases ancres par topic sont définies, maintenant on calcul leurs embeddings ainsi que ceux des phrases du corpus.

In [51]:
pip install sentence-transformers --break-system-packages

  Using cached sentence_transformers-5.4.1-py3-none-any.whl.metadata (17 kB)
  Using cached scikit_learn-1.8.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 45.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 71.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 81.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 77.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 77.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 712.2 kB/s  0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.3/801.3 kB 86.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 74.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22/22 [sentence-transformers]ence-transformers]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")

embedding_model = SentenceTransformer("dangvantuan/sentence-camembert-base", device=device)

# Embeddings du corpus
embeddings_corpus = embedding_model.encode(
    df["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f"Embeddings corpus : {embeddings_corpus.shape}")

# Embeddings des phrases-ancres
ancres_texts = list(phrases_ancres.values())
ancres_keys = list(phrases_ancres.keys())
embeddings_ancres = embedding_model.encode(
    ancres_texts,
    convert_to_numpy=True
)
ancre_embeddings = {key: emb for key, emb in zip(ancres_keys, embeddings_ancres)}
print(f"Embeddings ancres : {embeddings_ancres.shape}")

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/python/lib/python3.13/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Device : cpu


Batches:  23%|██▎       | 152/647 [03:58<08:17,  1.00s/it]

Maintenant on calcule les distances cosines entre les phrases types par topic et les phrases du corpus.

In [23]:
from sklearn.metrics.pairwise import cosine_similarity

# Pour chaque thème, calculer la similarité cosine entre chaque phrase et la phrase-ancre
for theme in themes:
    ancre = ancre_embeddings[theme].reshape(1, -1)
    similarities = cosine_similarity(embeddings_corpus, ancre).flatten()
    df[f"cosine_{theme}"] = similarities

print("Similarités cosine calculées !")
df[[f"cosine_{theme}" for theme in themes]].describe()

Similarités cosine calculées !


,cosine_forme_physique,cosine_force_caractere_effort,cosine_mental_leadership,cosine_emotion,cosine_apparence_physique,cosine_vie_personnelle,cosine_personnalite,cosine_chance,cosine_performance_heroisme,cosine_resultat_enjeux,cosine_age,cosine_marqueur_genre,cosine_violence,cosine_cooperation_equipe,cosine_analytique,cosine_encouragement,cosine_strategie
count,28459.000000,28459.000000,28459.000000,28459.000000,28459.000000,28459.000000,28459.000000,28459.000000,28459.000000,28459.000000,28459.000000,28459.000000,28459.000000,28459.000000,28459.000000,28459.000000,28459.000000
mean,0.140537,0.075682,0.078797,0.121114,0.100920,0.059805,0.059082,0.105267,0.165081,0.100254,0.113026,0.007851,0.127757,0.070415,0.083539,0.156035,0.073954
std,0.103925,0.078203,0.073561,0.091966,0.090304,0.070024,0.075910,0.076800,0.112783,0.121859,0.087674,0.087010,0.086458,0.091198,0.093809,0.091708,0.077202
min,-0.176108,-0.160557,-0.253980,-0.160989,-0.196787,-0.184862,-0.201592,-0.149145,-0.228057,-0.188807,-0.178745,-0.239980,-0.207316,-0.156368,-0.202374,-0.187661,-0.174190
25%,0.068586,0.023600,0.024306,0.061245,0.039481,0.012223,0.006615,0.051430,0.086206,0.018746,0.055204,-0.057535,0.066780,0.004389,0.019008,0.096402,0.023020
50%,0.134799,0.069585,0.072369,0.112797,0.093642,0.057302,0.056242,0.101874,0.165230,0.077232,0.104996,-0.003545,0.125024,0.053094,0.074821,0.151121,0.072142
75%,0.198246,0.114452,0.123233,0.170295,0.146709,0.099726,0.112196,0.157930,0.247497,0.153014,0.166841,0.056382,0.191152,0.115280,0.139071,0.208279,0.118254
max,0.673559,0.667281,0.530063,0.625487,0.614368,0.561992,0.561112,0.786017,0.704414,0.842781,0.627895,0.781098,0.588667,0.694935,0.700084,0.693879,0.683207


Vérification : est-ce que les différences (significatives) genrées trouvées avec la distance cosine correspondent à celles trouvées avec l'analyse en termes de prévalence ?

In [ ]:
import scipy.stats as stats

print("ATHLETES (H1)")
resultats_ath = []
for theme in themes:
    g_H = df[df["g_ath"] == "H"][f"cosine_{theme}"]
    g_F = df[df["g_ath"] == "F"][f"cosine_{theme}"]
    stat, pval = stats.mannwhitneyu(g_H, g_F, alternative="two-sided")
    resultats_ath.append({
        "theme": theme,
        "mean_H": g_H.mean(),
        "mean_F": g_F.mean(),
        "diff": g_H.mean() - g_F.mean(),
        "p_value": pval,
        "significatif": "*" if pval < 0.05 else ""
    })
pd.DataFrame(resultats_ath).sort_values("p_value").to_string()
print(pd.DataFrame(resultats_ath).sort_values("p_value").to_string())

print("\nJOURNALISTES (H2)")
resultats_jour = []
for theme in themes:
    g_H = df[df["main_g"] == "male"][f"cosine_{theme}"]
    g_F = df[df["main_g"] == "female"][f"cosine_{theme}"]
    stat, pval = stats.mannwhitneyu(g_H, g_F, alternative="two-sided")
    resultats_jour.append({
        "theme": theme,
        "mean_H": g_H.mean(),
        "mean_F": g_F.mean(),
        "diff": g_H.mean() - g_F.mean(),
        "p_value": pval,
        "significatif": "*" if pval < 0.05 else ""
    })
print(pd.DataFrame(resultats_jour).sort_values("p_value").to_string())

=== ATHLETES (H1) ===
                     theme    mean_H    mean_F      diff       p_value significatif
4       apparence_physique  0.082668  0.112757 -0.030089  0.000000e+00            *
5          vie_personnelle  0.054598  0.065755 -0.011157  7.092389e-36            *
9          resultat_enjeux  0.094740  0.109396 -0.014656  2.178035e-19            *
12                violence  0.121379  0.131854 -0.010475  4.564378e-18            *
15           encouragement  0.149058  0.158805 -0.009746  7.324355e-16            *
11          marqueur_genre  0.005038  0.014673 -0.009635  4.695548e-15            *
16               strategie  0.072332  0.078709 -0.006378  9.634168e-11            *
8     performance_heroisme  0.158644  0.167368 -0.008725  1.125726e-08            *
10                     age  0.109208  0.114911 -0.005704  1.038875e-06            *
2        mental_leadership  0.076759  0.080737 -0.003979  4.857744e-06            *
7                   chance  0.107724  0.104449  0.0032